In [0]:
# HC_Practice Week 2 - Monday
# Databricks Notebook - PySpark Exploration
# Charles Richardson | June 23, 2026
# Goal: Read claims CSV into a Spark DataFrame and run basic exploration

print("Notebook initialized")

Notebook initialized


In [0]:
# Read claims CSV from Unity Catalog Volume into Spark DataFrame
df = spark.read.csv(
    "/Volumes/workspace/default/hc_practice_data/new_claims_DB.csv", 
    header=True, 
    inferSchema=True
)

# Equivalent to SELECT TOP 5 * FROM claims
df.show(5)

# Equivalent to sp_help 'claims' -- shows column names and data types
df.printSchema()

+--------+---------+-----------+------------+----------+--------------+--------------+--------------+-------------+--------------+-----------+------------+--------------------+--------------+
|claim_id|member_id|provider_id|service_date|admit_date|discharge_date|diagnosis_code|procedure_code|billed_amount|allowed_amount|paid_amount|claim_status|       denial_reason|load_timestamp|
+--------+---------+-----------+------------+----------+--------------+--------------+--------------+-------------+--------------+-----------+------------+--------------------+--------------+
|    1114|        5|          3|  2026-06-01|2026-06-01|    2026-06-03|         J18.9|         99213|         4200|          3800|       3800|    Approved|                NULL|    2026-06-22|
|    1115|       12|          7|  2026-06-03|2026-06-03|    2026-06-03|         M54.5|         99214|          950|           800|        800|    Approved|                NULL|    2026-06-22|
|    1116|       19|          2|  2026-0

In [0]:
# Read claims CSV into DataFrame
df = spark.read.csv(
    "/Volumes/workspace/default/hc_practice_data/new_claims_DB.csv",
    header=True,
    inferSchema=True
)

# Count denied claims - equivalent to:
# SELECT COUNT(*) FROM claims WHERE claim_status = 'Denied'
denied_count = df.filter(df.claim_status == "Denied").count()

print(f"Total Denied Claims: {denied_count}")

Total Denied Claims: 4


In [0]:
# Summary statistics - equivalent to a SQL profiling query
# Shows count, mean, stddev, min, max for numeric columns
df = spark.read.csv(
    "/Volumes/workspace/default/hc_practice_data/new_claims_DB.csv",
    header=True,
    inferSchema=True
)
df.describe().show()


+-------+------------------+------------------+------------------+--------------+------------------+-----------------+-----------------+-----------------+------------+-------------------+
|summary|          claim_id|         member_id|       provider_id|diagnosis_code|    procedure_code|    billed_amount|   allowed_amount|      paid_amount|claim_status|      denial_reason|
+-------+------------------+------------------+------------------+--------------+------------------+-----------------+-----------------+-----------------+------------+-------------------+
|  count|                10|                10|                10|            10|                10|               10|               10|               10|          10|                  4|
|   mean|            1118.5|              10.7|               5.5|          NULL|           91216.3|           4099.0|           2717.0|           2717.0|        NULL|               NULL|
| stddev|3.0276503540974917|6.8645627844912465|3.02765035409

In [0]:
# Denial Rate by Provider - PySpark equivalent of denial_rate_by_provider_and_specialty
# SQL equivalent: SELECT provider_id, COUNT(*), SUM(CASE WHEN claim_status='Denied'...) GROUP BY provider_id

from pyspark.sql.functions import count, sum, when, round

df = spark.read.csv(
    "/Volumes/workspace/default/hc_practice_data/new_claims_DB.csv",
    header=True,
    inferSchema=True
)

denial_rate = df.groupBy("provider_id") \
    .agg(
        count("claim_id").alias("total_claims"),
        sum(when(df.claim_status == "Denied", 1).otherwise(0)).alias("denied_claims"),
        round(
            sum(when(df.claim_status == "Denied", 1).otherwise(0)) * 100.0 
            / count("claim_id"), 2
        ).alias("denial_rate_pct")
    ) \
    .orderBy("denial_rate_pct", ascending=False)

denial_rate.show()

+-----------+------------+-------------+---------------+
|provider_id|total_claims|denied_claims|denial_rate_pct|
+-----------+------------+-------------+---------------+
|          5|           1|            1|          100.0|
|          2|           1|            1|          100.0|
|          9|           1|            1|          100.0|
|          4|           1|            1|          100.0|
|         10|           1|            0|            0.0|
|          3|           1|            0|            0.0|
|          1|           1|            0|            0.0|
|          6|           1|            0|            0.0|
|          7|           1|            0|            0.0|
|          8|           1|            0|            0.0|
+-----------+------------+-------------+---------------+



In [0]:
# Member Utilization - PySpark equivalent of member_utilization query
# SQL: SELECT member_id, COUNT(DISTINCT claim_id), SUM(billed_amount) GROUP BY member_id

from pyspark.sql.functions import countDistinct, sum, round

df = spark.read.csv(
    "/Volumes/workspace/default/hc_practice_data/new_claims_DB.csv",
    header=True,
    inferSchema=True
)

member_util = df.groupBy("member_id") \
    .agg(
        countDistinct("claim_id").alias("total_claims"),
        round(sum("billed_amount"), 2).alias("total_billed")
    ) \
    .orderBy("total_billed", ascending=False)

member_util.show()

+---------+------------+------------+
|member_id|total_claims|total_billed|
+---------+------------+------------+
|       14|           1|       15200|
|        2|           1|        9400|
|        8|           1|        7800|
|        5|           1|        4200|
|       17|           1|        1100|
|       12|           1|         950|
|        6|           1|         890|
|        3|           1|         620|
|       21|           1|         480|
|       19|           1|         350|
+---------+------------+------------+



In [0]:
# Read JSON file into Spark DataFrame using spark.read.json()
# Equivalent to OPENJSON() in SQL Server

df_json = spark.read.json(
    "/Volumes/workspace/default/hc_practice_data/claims_data.json"
)

# Show first 5 rows
df_json.show(5)

# Check schema - Spark automatically detects JSON structure
df_json.printSchema()

+-------------+--------+------------+--------------------+--------------+---------+-----------+------------+
|billed_amount|claim_id|claim_status|       denial_reason|diagnosis_code|member_id|provider_id|service_date|
+-------------+--------+------------+--------------------+--------------+---------+-----------+------------+
|       4200.0|    1114|    Approved|                NULL|         J18.9|        5|          3|  2026-06-01|
|        950.0|    1115|    Approved|                NULL|         M54.5|       12|          7|  2026-06-03|
|        350.0|    1116|      Denied|Not Medically Nec...|        Z00.00|       19|          2|  2026-06-05|
|       7800.0|    1117|    Approved|                NULL|           I10|        8|         10|  2026-06-07|
|        620.0|    1118|      Denied| Prior Auth Required|         E11.9|        3|          5|  2026-06-08|
+-------------+--------+------------+--------------------+--------------+---------+-----------+------------+
only showing top 5 

In [0]:
# Write claims data to Delta table in Unity Catalog
df = spark.read.csv(
    "/Volumes/workspace/default/hc_practice_data/new_claims_DB.csv",
    header=True,
    inferSchema=True
)

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.claims")

print("Delta table 'workspace.default.claims' created successfully.")

Delta table 'workspace.default.claims' created successfully.
